# Special Days → `special_events` lakehouse

Run this on the THY Spark nonprod cluster (JupyterHub, XS) **after `git pull`**.
It scrapes the next 12 months of special dates, enriches them (heuristic scorer,
no LLM), and writes two Hive/Parquet tables under `/lakehouse/special_events`:

| Table | Grain |
| --- | --- |
| `special_events.special_days_raw` | one row per special date (span grain) |
| `special_events.special_days_features` | one row per `(event_date, country, airport)` |

Start the kernel from the **repo root** (or anywhere inside it — the first cell
walks up to find `special_days/`). Source API keys are optional: set them in a
`.env` at the repo root for Ticketmaster / API-Football / EventsEye; without
them you still get all the holiday sources.

In [ ]:
# Put the repo root on sys.path so `import special_days` works from the notebook.
import os, sys
cur = os.getcwd()
while cur != os.path.dirname(cur):
    if os.path.isdir(os.path.join(cur, 'special_days')):
        if cur not in sys.path:
            sys.path.insert(0, cur)
        break
    cur = os.path.dirname(cur)
print('repo root:', cur)

In [ ]:
# A Spark session may already be provided as `spark`; getOrCreate() reuses it.
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('special-days-lakehouse')
    .enableHiveSupport()
    .getOrCreate()
)
spark

In [ ]:
# Scrape + enrich the next 12 months (heuristic scorer — no live LLM calls).
from special_days.agents import TurkeyAgent, InternationalAgent
from special_days.enrich import enrich, drop_long_events, DEFAULT_MAX_EVENT_SPAN_DAYS
from special_days.scoring import HeuristicScorer
from special_days.window import resolve_window
from special_days.models import SpecialDate
from special_days.config import load_dotenv

load_dotenv()  # optional: source API keys from a .env at the repo root
start, end = resolve_window(None, 12)  # today .. +12 months

records = []
for agent in (TurkeyAgent(), InternationalAgent()):
    records.extend(agent.collect(start, end, include_holidays=True, include_events=True))
records = list(dict.fromkeys(records))                 # de-dup overlap
records = drop_long_events(records, DEFAULT_MAX_EVENT_SPAN_DAYS)
records = enrich(records, scorer=HeuristicScorer())
records.sort(key=SpecialDate.sort_key)
print(f'{len(records)} special date(s) {start} -> {end}')

In [ ]:
# Write both tables (full overwrite — idempotent at this volume).
from special_days.sinks import lakehouse
run_id = lakehouse.write(
    records,
    spark=spark,
    database='special_events',
    location='/lakehouse/special_events',
)
print('run_id:', run_id)

In [ ]:
# Verify.
spark.sql('SELECT COUNT(*) AS raw_rows FROM special_events.special_days_raw').show()
spark.sql('SELECT COUNT(*) AS feature_rows FROM special_events.special_days_features').show()
spark.sql('''
    SELECT event_date, country, airport, impact, predicted_attendance, sources, n_events
    FROM special_events.special_days_features
    ORDER BY event_date
    LIMIT 20
''').show(truncate=False)